# Día 3 · Experimento HyDE con BGE-M3

Compara la recuperación usando la pregunta original frente a un documento hipotético. Ambos métodos usan los mismos 345 chunks, BGE-M3, top-5 y juez de relevancia.

## 1. Cargar el proyecto y dependencias

Esta celda trabaja con la rama del Día 3 en GitHub. No modifica tu computador.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-03-experimento-hyde
!pip -q install pandas pyarrow openpyxl sentence-transformers scikit-learn openai

## 2. Registrar la clave de forma privada

La clave no se muestra, no se guarda en el notebook y no se sube a GitHub.

In [ ]:
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
print('Clave cargada temporalmente en esta sesión.')

## 3. Ejecutar el experimento

Se harán aproximadamente 20 llamadas para HyDE y 20 para evaluar el pool. Los checkpoints permiten reanudar si Colab se interrumpe. Las tarifas son parámetros editables; el resultado guarda tokens reales y costo estimado.

In [ ]:
!python src/retrieval/run_hyde_experiment.py \
  --embeddings data/processed/embedding/embeddings_bge_m3.parquet \
  --questions data/evaluation/gold_questions.csv \
  --output-dir /content/resultados_hyde_dia_03 \
  --top-k 5 \
  --review-size 30 \
  --llm-model gpt-4o-mini \
  --input-cost-per-million 0.15 \
  --output-cost-per-million 0.60

## 4. Mostrar los resultados

Precision indica cuántos resultados recuperados fueron relevantes. Hit indica en cuántas preguntas apareció al menos un resultado relevante. MRR premia que el primer resultado relevante aparezca en una posición alta.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

out = Path('/content/resultados_hyde_dia_03')
metrics = pd.read_csv(out / 'metrics_by_method.csv')
metadata = json.loads((out / 'experiment_metadata.json').read_text(encoding='utf-8'))
display(metrics)
display(pd.DataFrame([metadata]))

## 5. Descargar el paquete

Descarga el ZIP y compártelo en el chat. Con él consolidaremos los resultados, revisaremos si HyDE mejora y cerraremos el Día 3.

In [ ]:
import shutil
from google.colab import files

zip_result = shutil.make_archive('/content/resultados_hyde_dia_03', 'zip', out)
files.download(zip_result)